# NetWorld-AI: Phase 1 — Exploratory Data Analysis (EDA)
**SIH 2026 NTRO PS 26153**

This notebook inspects the synthetic sample dataset (`data/sample/sample_cic_ids.csv`) mimicking the CIC-IDS2018 benchmark schema. It analyzes dataset dimensions, column distributions, missing/infinite values, class imbalance, and feature correlation.

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Ensure project root is in working directory
print("Current Directory:", os.getcwd())

## 1. Load Dataset

In [ ]:
csv_path = "data/sample/sample_cic_ids.csv"
if not os.path.exists(csv_path):
    from src.generate_sample_data import generate_sample_cic_ids
    generate_sample_cic_ids()

df = pd.read_csv(csv_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Check Data Integrity: Missing & Infinite Values

In [ ]:
# Strip whitespace from column names
df.columns = df.columns.str.strip()

missing_counts = df.isnull().sum()
missing_total = missing_counts.sum()

# Check for infinite values in numeric columns
num_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[num_cols]).sum().sum()

print(f"Total Missing Values (NaN): {missing_total}")
print(f"Total Infinite Values (Inf/-Inf): {inf_counts}")
print(f"Duplicate Rows: {df.duplicated().sum()}")

## 3. Class Imbalance & Label Distribution

In [ ]:
label_counts = df["Label"].value_counts().reset_index()
label_counts.columns = ["Label", "Count"]
label_counts["Percentage"] = (label_counts["Count"] / len(df)) * 100
print(label_counts)

fig = px.bar(
    label_counts, 
    x="Label", 
    y="Count", 
    text="Percentage",
    title="Class Distribution in Sample Dataset",
    color="Label"
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.show()

## 4. Feature Extraction & Cleaning with CSVLoader

In [ ]:
from src.data.csv_loader import CSVLoader

loader = CSVLoader(csv_path)
raw_df = loader.load_raw()
extracted_df, feature_cols = loader.extract_features(raw_df)
cleaned_df = loader.clean_data(extracted_df)

print(f"Extracted Feature DataFrame Shape: {cleaned_df.shape}")
print("Extracted Features List:", feature_cols)
cleaned_df[feature_cols].describe()